# netmotion in ten minutes

Build networks, measure them, and run the processes of the book *Networks in Motion* on them: opinions, an infection, immunisation and a banking crisis.

Every function takes named parameters and a random seed, `rng`, so each cell gives the same result every time. Change a number and run the cell again.

## 1. Build three networks

A random graph (Erdős–Rényi), a small world (Watts–Strogatz: a ring with a few links rewired) and a scale-free network (Barabási–Albert: each new node prefers the best-connected ones).

In [ ]:
from netmotion.networks import barabasi_albert, erdos_renyi, watts_strogatz

random_graph = erdos_renyi(n_nodes=200, link_probability=0.03, rng=1)
small_world = watts_strogatz(n_nodes=200, neighbours=6, rewiring_probability=0.05, rng=1)
scale_free = barabasi_albert(n_nodes=200, links_per_new_node=2, initial_nodes=3, rng=1)
small_world

## 2. Measure them

The mean degree ⟨k⟩, the mean distance L between two nodes, and the clustering coefficient C. The small world keeps a high C with a short L; the scale-free network has a degree distribution that falls as a power law.

In [ ]:
from netmotion.metrics import clustering_coefficient, mean_degree, mean_distance, power_law_exponent

for name, g in [("random graph", random_graph), ("small world", small_world), ("scale free", scale_free)]:
    print(f"{name:13} <k> = {mean_degree(g):.2f}   L = {mean_distance(g):.2f}   C = {clustering_coefficient(g):.3f}")
print(f"scale free: P(k) ~ k^-{power_law_exponent(scale_free, min_degree=2):.2f}")

## 3. Opinions

Hopfield dynamics: every node holds +1 (in favour) or −1 (against) and, at each step, takes the sign of the sum of its neighbours' opinions.

In [ ]:
from netmotion.dynamics import random_initial_states, run_hopfield

start = random_initial_states(n_nodes=200, n_against=90, rng=2)   # 90 nodes start at -1
run = run_hopfield(small_world, start, steps=12)
print("in favour at each step:", run.n_plus.tolist())

## 4. An infection

At each instant every infected node infects all its neighbours. The speed of the spread is the factor α of p(t) = A₀ e^(αt), fitted to the fraction infected.

In [ ]:
from netmotion.propagation import mean_propagation_speed, propagate, propagation_speed

infection = propagate(small_world, first_infected=0)
print(f"reaches everyone in {infection.steps} steps; alpha = {propagation_speed(infection.curve):.2f}")
print(f"mean alpha over every starting node: {mean_propagation_speed(small_world):.2f}")

## 5. Which nodes matter

The degree of order n, K(n), counts the links a node reaches within n steps, not just its own. The central node is the first to reach the whole network.

In [ ]:
from netmotion.metrics import central_node, k_order, k_order_ranking

print("K(2) of node 0:", k_order(scale_free, node=0, order=2))
print("top five by K(2):", k_order_ranking(scale_free, order=2)[:5])
print("central node:", central_node(scale_free))

## 6. Immunise

Make the top nodes by K(2) immune, a few at a time, and watch the mean speed of the infection fall. β is the rate at which it falls.

In [ ]:
from netmotion.propagation import immunisation_sweep

sweep = immunisation_sweep(scale_free, criterion="k_order_n", order=2, counts=range(0, 21, 2), rng=3)
for immune, alpha in zip(sweep.counts, sweep.speed):
    print(f"{immune:3} immune nodes -> mean alpha {alpha:.3f}")
print(f"beta = {sweep.beta:.3f}")

## 7. A banking crisis

Six countries grow bank by bank with the seven strategies of the book; the links are loans. Then the largest bank fails. Each bank keeps reserves (here 20% of its balance) and calls in its loans once it has lost 30% of its capital.

In [ ]:
from netmotion.finance import InterbankNetwork, bank_capital, four_state_cascade, strategy_growth

grown, strategies = strategy_growth(n_communities=6, steps=120, rng=7)   # 6 countries
banks = InterbankNetwork.from_community_network(grown)
crash = four_state_cascade(banks.network, banks.largest_bank(), bank_capital(banks.network),
                           reserves=20.0, reaction_threshold=30.0)
print(f"{banks.n_banks} banks, {len(banks.too_big_to_fail())} too big to fail")
print(f"the largest fails: {crash.crashed_fraction:.0%} of the banks fall in {crash.steps} steps")

## 8. Watch it

Press ▶ under the picture. Standing banks are green, brown or purple by their balance; they turn red when they fall.

In [ ]:
from netmotion.visualization import animate_cascade, to_html

to_html(animate_cascade(banks, crash))

## Next

- `animations.ipynb`: growth, infection and opinions animated, and an animation of your own.
- [networks-in-motion-tutorials](https://github.com/marianotir/networks-in-motion-tutorials): the book chapter by chapter, in 14 notebooks.
- `python -m netmotion.experiments.cli --list`: every figure of the book, drawn again from its seed.